# 07장 조건부 확률과 베이즈 — 새 정보로 고치는 믿음 — 수리 도감 코랩 실습

원자료: 연구소 수학 노트 MATH-07 「Conditional Probability & Bayes' Theorem」 · 해설 페이지: https://adminmrobo.github.io/math_basic_for_ai_dictionary/07_bayes.html

위에서부터 셀을 차례로 실행하세요(Shift + Enter). 그래프는 영어 글자로 그렸습니다(코랩에 한글 글꼴을 따로 깔지 않아도 되게).

글 · 안상선 ((주)M-Robo 대표)


## 1. 베이즈 정리 함수


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def bayes(prior, sens, fpr):
    """P(병|양성) = P(양성|병)P(병) / P(양성)"""
    evidence = sens * prior + fpr * (1 - prior)     # 전확률: 양성이 나오는 모든 길
    return sens * prior / evidence

p = bayes(0.01, 0.99, 0.05)
print(f"P(병|양성) = 0.0099 / (0.0099 + 0.0495) = {p:.4f}  (약 {p*100:.1f}%)")

## 2. 가상 인구로 확인


In [ ]:
rng = np.random.default_rng(7)
for N in [1000, 1_000_000]:
    sick = rng.random(N) < 0.01
    positive = np.where(sick, rng.random(N) < 0.99, rng.random(N) < 0.05)
    tp, fp = np.sum(sick & positive), np.sum(~sick & positive)
    fn, tn = np.sum(sick & ~positive), np.sum(~sick & ~positive)
    print(f"N={N:>9,}: 진양성 {tp}, 거짓양성 {fp}, 거짓음성 {fn}, 진음성 {tn} -> 양성 중 환자 비율 {tp/(tp+fp):.3f}")

## 3. 유병률에 따라 달라지는 사후확률


In [ ]:
priors = np.linspace(0.001, 0.5, 300)
plt.plot(priors * 100, [bayes(q, 0.99, 0.05) for q in priors])
plt.axvline(1, color="gray", ls=":"); plt.xlabel("prevalence (%)"); plt.ylabel("P(disease | positive)")
plt.title("Same test, different base rates"); plt.grid(alpha=0.3); plt.show()

## 4. 베이즈 갱신: 사후확률이 다음 사전확률


In [ ]:
belief = 0.01
for k in range(1, 4):
    belief = bayes(belief, 0.99, 0.05)
    print(f"양성 {k}번 뒤: P(병) = {belief:.3f}")

## 5. 비복원 추출: 빨강 5, 파랑 3


In [ ]:
bag = np.array(["R"] * 5 + ["B"] * 3)
first_red = second_red_too = 0
for _ in range(100000):
    draw = rng.choice(bag, size=2, replace=False)
    if draw[0] == "R":
        first_red += 1
        second_red_too += draw[1] == "R"
print(f"P(둘째 빨강 | 첫째 빨강) ≈ {second_red_too/first_red:.3f} (이론 4/7 = {4/7:.3f})")

## 생성 AI 실습 프롬프트

아래 글을 코랩의 Gemini 창(또는 다른 생성 AI)에 붙여 넣고, 만들어진 코드를 위 코드의 결과와 비교해 보세요.

```text
너는 고등학생에게 베이즈 정리를 가르치는 파이썬 선생님이야. 구글 코랩에서 실행되는 코드를 만들어 줘.
주제: 조건부 확률과 베이즈 정리(의료 검사 예시)
요구 사항:
1. 함수 bayes(prior, sensitivity, false_positive_rate)를 만들어 P(병|양성)을 계산하고, 유병률 1%, 민감도 99%, 거짓 양성 5%일 때 약 0.167이 나오는지 출력해 줘.
2. 1000명 가상 인구를 numpy로 시뮬레이션(seed 고정)해서 진양성·거짓양성·진음성·거짓음성 인원을 세고, 시뮬레이션으로 구한 P(병|양성)을 공식 값과 비교해 줘.
3. 유병률을 0.1%부터 50%까지 바꿔 가며 P(병|양성)이 어떻게 변하는지 선 그래프로 그려 줘(영어 라벨).
4. 양성이 두 번 연속 나왔을 때(두 검사는 독립이라고 가정) 사후확률을 다시 사전확률로 넣어 갱신하는 과정을 출력해 줘.
5. 빨강 5개·파랑 3개 주머니에서 비복원으로 두 개 뽑는 시뮬레이션으로 P(둘째 빨강|첫째 빨강) ≈ 4/7을 확인해 줘.
6. 모든 줄에 한국어 주석.
```
